# Data Pre-processing for Transformer-Based Sentiment Classification - CUSTOMER ONLY

**Important NOTE:** This notebook prepares the dataset for a transformer model.
Since transformers work directly with **contextual text**, the preprocessing is kept **minimal**.

This notebook is a replica of the **pre-process.ipynb** notebook. The only additional operation is that during preprocessing, we keep only the customer text from the **conversation** column.

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

/Users/fatmaoztel/Desktop/Fatima Projects/cs-conversation-sentiment-analysis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the data

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (970, 11)
Test shape: (30, 11)


First load the train and test sets so that the preprocessing pipeline is applied consistently.

## Select target and modeling columns

In [3]:
target_col = "customer_sentiment"
text_col = "conversation"

selected_columns = [text_col, target_col]

train_model_df = train_df[selected_columns].copy()
test_model_df = test_df[selected_columns].copy()

print("Train modeling shape:", train_model_df.shape)
print("Test modeling shape:", test_model_df.shape)
display(train_model_df.head())

Train modeling shape: (970, 2)
Test modeling shape: (30, 2)


,conversation,customer_sentiment
0,"Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?\n\nCustomer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG),...",neutral
1,Agent: Thank you for calling BrownBox customer support. My name is Alex. How may I assist you today?\n\nCustomer: Hi Alex. I recently received an email from BrownBox requesting me to ship back the...,neutral
2,"Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How may I assist you today?\n\nCustomer: Hi Sarah, I am calling because I am unable to click the 'Cancel' button for my Ju...",neutral
3,"Customer: Hi, I am facing an issue while logging into my account. I am getting an error message saying that I have exceeded the number of attempts to enter the correct verification code.\n\nAgent:...",neutral
4,"Agent: Thank you for contacting BrownBox customer support. My name is Sarah. How can I assist you today?\n\nCustomer: Hi Sarah, I have an issue with my order. I received a BP monitor, but the deli...",negative


Use only the "conversation" column as model input, because transformers are designed for text sequences. Structured columns such as "issue_category" clearly defined and analyzed on the EDA Notebook may still be useful, but adding them would require a more complex hybrid architecture. Let's do that for improvement the available model LATER.

## Check missing values

In [4]:
train_missing = train_model_df.isnull().sum().sort_values(ascending=False)
test_missing = test_model_df.isnull().sum().sort_values(ascending=False)

print("Train missing values:")
display(train_missing[train_missing > 0])

print("Test missing values:")
display(test_missing[test_missing > 0])

Train missing values:


Series([], dtype: int64)

Test missing values:


Series([], dtype: int64)

Missing values are checked before tokenization because null text values would cause errors in later steps. No missing values are found, so no imputation is needed.

## Handle duplicate rows

In [5]:
print("Duplicate rows in train_model_df:", train_model_df.duplicated().sum())
print("Duplicate conversation texts in train_model_df:", train_model_df[text_col].duplicated().sum())

Duplicate rows in train_model_df: 2
Duplicate conversation texts in train_model_df: 2


In [6]:
train_model_df = train_model_df.drop_duplicates().reset_index(drop=True)

print("Train shape after removing duplicate rows:", train_model_df.shape)

Train shape after removing duplicate rows: (968, 2)


Exact duplicate rows were removed. It was detected on the EDA nb as well.

## Encode sentiment labels

In [7]:
label_mapping = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_model_df["sentiment_label"] = train_model_df[target_col].map(label_mapping)
test_model_df["sentiment_label"] = test_model_df[target_col].map(label_mapping)

print("Label mapping:", label_mapping)
display(train_model_df[[target_col, "sentiment_label"]].drop_duplicates().sort_values("sentiment_label"))

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}


,customer_sentiment,sentiment_label
4,negative,0
0,neutral,1
143,positive,2


Labels are converted to numeric values because transformer classifiers require numerical targets during training.

## Create a validation set

In [8]:
train_split_df, val_split_df = train_test_split(
    train_model_df,
    test_size=0.2,
    random_state=42,
    stratify=train_model_df["sentiment_label"]
)

train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

print("Train split shape:", train_split_df.shape)
print("Validation split shape:", val_split_df.shape)

Train split shape: (774, 3)
Validation split shape: (194, 3)


In [9]:
print("Training set sentiment distribution:")
(train_split_df[target_col].value_counts(normalize=True) * 100).round(2)

Training set sentiment distribution:


customer_sentiment
neutral     55.94
negative    42.25
positive     1.81
Name: proportion, dtype: float64

In [10]:
print("Validation set sentiment distribution:")
(val_split_df[target_col].value_counts(normalize=True) * 100).round(2)

Validation set sentiment distribution:


customer_sentiment
neutral     56.19
negative    42.27
positive     1.55
Name: proportion, dtype: float64

80-20 split has been chosen. 80-20 split is used to keep most samples for training while reserving a meaningful validation set. Stratified splitting is chosen because the dataset is imbalanced, especially for the positive class.

## Extract only customer text

In [11]:
def extract_customer_only_text(conversation):
    conversation = str(conversation)
    customer_parts = re.findall(r"Customer:\s*(.*?)(?=\n\n(?:Agent|Customer):|$)", conversation, flags=re.DOTALL)
    customer_parts = [part.strip() for part in customer_parts if part.strip()]
    return " ".join(customer_parts)

In [12]:
train_split_df["customer_only_text"] = train_split_df[text_col].apply(extract_customer_only_text)
val_split_df["customer_only_text"] = val_split_df[text_col].apply(extract_customer_only_text)
test_model_df["customer_only_text"] = test_model_df[text_col].apply(extract_customer_only_text)

display(train_split_df[[text_col, "customer_only_text"]].head(3))

,conversation,customer_only_text
0,"Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How can I assist you today?\n\nCustomer: Hi Sarah, I have a question about the warranty for a water geyser I purchased fro...","Hi Sarah, I have a question about the warranty for a water geyser I purchased from BrownBox. My order number is 789012 and I purchased it on the 15th of last month. Okay, that's good to know. But ..."
1,"Customer: Hi, I purchased a Smart Watch from your website a few months ago, but I can't seem to remember the warranty details. Can you help me with that?\n\nAgent: Hello! I'm sorry to hear that yo...","Hi, I purchased a Smart Watch from your website a few months ago, but I can't seem to remember the warranty details. Can you help me with that? Sure, my order number is BB123456789. No, not really..."
2,"Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?\n\nCustomer: Hi Tom, I recently bought a pair of shorts from your website, but they don't fit me...","Hi Tom, I recently bought a pair of shorts from your website, but they don't fit me well. I want to return them. Sure, my order number is BB123456, and I want to return the shorts as they don't fi..."


This step keeps only the customer part/speech from the conversation so the model focuses more directly on the customer's side of the interaction.

See exact example:

In [13]:
sample_idx = 0

print("FULL CONVERSATION:\n")
print(train_split_df.loc[sample_idx, text_col])

print("\n" + "="*100 + "\n")

print("CUSTOMER ONLY TEXT:\n")
print(train_split_df.loc[sample_idx, "customer_only_text"])

FULL CONVERSATION:

Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How can I assist you today?

Customer: Hi Sarah, I have a question about the warranty for a water geyser I purchased from BrownBox.

Agent: Sure, I'll be happy to help you with that. Can you please provide me with your order number and the date of purchase?

Customer: My order number is 789012 and I purchased it on the 15th of last month.

Agent: Thank you for the information. Let me check the details of your purchase. (Agent checks the details) According to our records, your warranty starts from the date of purchase, which is the 15th of last month.

Customer: Okay, that's good to know. But I have a question. I didn't install the geyser until a week later. Does that mean my warranty starts from the date of installation?

Agent: That's a good question. Let me check with our team to confirm. Please hold on for a moment.

(Customer is on hold for a minute.)

Agent: Thank you for holding. I have 

## Light text normalization - Continue with Customer text only

In [14]:
def normalize_text_for_transformer(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

train_split_df["conversation_clean"] = train_split_df["customer_only_text"].apply(normalize_text_for_transformer)
val_split_df["conversation_clean"] = val_split_df["customer_only_text"].apply(normalize_text_for_transformer)
test_model_df["conversation_clean"] = test_model_df["customer_only_text"].apply(normalize_text_for_transformer)

display(train_split_df[["customer_only_text", "conversation_clean"]].head(3))

,customer_only_text,conversation_clean
0,"Hi Sarah, I have a question about the warranty for a water geyser I purchased from BrownBox. My order number is 789012 and I purchased it on the 15th of last month. Okay, that's good to know. But ...","hi sarah, i have a question about the warranty for a water geyser i purchased from brownbox. my order number is 789012 and i purchased it on the 15th of last month. okay, that's good to know. but ..."
1,"Hi, I purchased a Smart Watch from your website a few months ago, but I can't seem to remember the warranty details. Can you help me with that? Sure, my order number is BB123456789. No, not really...","hi, i purchased a smart watch from your website a few months ago, but i can't seem to remember the warranty details. can you help me with that? sure, my order number is bb123456789. no, not really..."
2,"Hi Tom, I recently bought a pair of shorts from your website, but they don't fit me well. I want to return them. Sure, my order number is BB123456, and I want to return the shorts as they don't fi...","hi tom, i recently bought a pair of shorts from your website, but they don't fit me well. i want to return them. sure, my order number is bb123456, and i want to return the shorts as they don't fi..."


After extracting customer-only text, only light normalization is applied. We keep the original wording and sequence structure because transformers learn context from the text itself. We do not remove stopwords, stem, or lemmatize.

## Load the pretrained transformer tokenizer

In [15]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Tokenizer loaded from:", model_checkpoint)

Tokenizer loaded from: distilbert-base-uncased


A pretrained tokenizer is used because transformer models already come with their own vocabulary and tokenization rules. This is more appropriate than creating a custom tokenizer. --> No need to create from scratch.

The tokenizer for distilbert-base-uncased converts raw text into the numerical format DistilBERT can understand.

What it does:

* lowercases the text: Because the model is uncased, it treats uppercase and lowercase the same.
* splits text into tokens: It breaks the sentence into smaller units, often words or subwords.
* uses subword tokenization: If a word is rare or unknown, it can split it into smaller pieces instead of failing completely.
* maps tokens to IDs: Each token is converted into an integer from the model vocabulary.
* adds special tokens: It adds tokens such as [CLS] and [SEP] if needed by the model format.
* creates attention masks: It marks which parts are real text and which parts are padding.
* pads and truncates sequences: it makes all inputs the same length for batch processing.

## Check token length

In [16]:
token_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in train_split_df["conversation_clean"]]

print("Mean token length:", round(np.mean(token_lengths), 2))
print("Median token length:", round(np.median(token_lengths), 2))
print("95th percentile:", int(np.percentile(token_lengths, 95)))
print("Max token length:", np.max(token_lengths))

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


Mean token length: 161.6
Median token length: 142.0
95th percentile: 386
Max token length: 927


In [17]:
max_length = min(512, int(np.percentile(token_lengths, 95)))
print("Chosen max_length:", max_length)

Chosen max_length: 386


The maximum sequence length is chosen from the training distribution and capped at 512 tokens due to model restriction.

We did that to balance two things:
- keeping enough of each conversation
- avoiding unnecessary computation

Why not use the full maximum length?
- some conversations are much longer than most others
- if we pad everything to the longest conversation, training becomes slower, memory usage increases a lot
- most rows would contain too much padding
- the 95th percentile keeps most samples almost complete
BUT in this case, due to model restriction, we will use 512 token

Why use the training distribution? (IMPORTANT FOR NO DATA LEAKAGE)
- it shows what a typical conversation length looks like by choosing from the training set avoids using information from validation/test unnecessarily

Should we cap at less than 512?
- transformer cost grows with sequence length: longer sequences require much more memory and time
- if memory becomes a problem, we can try 384 --> LETS SEE

## Tokenize the text

In [18]:
train_encodings = tokenizer(
    train_split_df["conversation_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="np"
)

val_encodings = tokenizer(
    val_split_df["conversation_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="np"
)

test_encodings = tokenizer(
    test_model_df["conversation_clean"].tolist(),
    truncation=True,
    padding=True,
    max_length=max_length,
    return_tensors="np"
)

print("Train input_ids shape:", train_encodings["input_ids"].shape)
print("Validation input_ids shape:", val_encodings["input_ids"].shape)
print("Test input_ids shape:", test_encodings["input_ids"].shape)

Train input_ids shape: (774, 386)
Validation input_ids shape: (194, 386)
Test input_ids shape: (30, 386)


input_ids and attention_mask are the main inputs a transformer needs.

**input_ids:**
- these are the token numbers produced by the tokenizer
- each word or subword is converted into an integer ID from the model vocabulary
- this is how the model reads the text

**attention_mask:**
- this tells the model which positions are real text and which are just padding
- 1 means real token
- 0 means padded position

## Prepare final model inputs

In [19]:
X_train_input_ids_customer_only = train_encodings["input_ids"]
X_train_attention_mask_customer_only = train_encodings["attention_mask"]

X_val_input_ids_customer_only = val_encodings["input_ids"]
X_val_attention_mask_customer_only = val_encodings["attention_mask"]

X_test_input_ids_customer_only = test_encodings["input_ids"]
X_test_attention_mask_customer_only= test_encodings["attention_mask"]

y_train_customer_only = train_split_df["sentiment_label"].values
y_val_customer_only = val_split_df["sentiment_label"].values
y_test_customer_only = test_model_df["sentiment_label"].values

print("X_train_input_ids_customer_only shape:", X_train_input_ids_customer_only.shape)
print("X_train_attention_mask_customer_only shape:", X_train_attention_mask_customer_only.shape)
print("X_val_input_ids_customer_only shape:", X_val_input_ids_customer_only.shape)
print("X_test_input_ids_customer_only shape:", X_test_input_ids_customer_only.shape)
print("y_train_customer_only shape:", y_train_customer_only.shape)
print("y_val_customer_only shape:", y_val_customer_only.shape)
print("y_test_customer_only shape:", y_test_customer_only.shape)

X_train_input_ids_customer_only shape: (774, 386)
X_train_attention_mask_customer_only shape: (774, 386)
X_val_input_ids_customer_only shape: (194, 386)
X_test_input_ids_customer_only shape: (30, 386)
y_train_customer_only shape: (774,)
y_val_customer_only shape: (194,)
y_test_customer_only shape: (30,)


These arrays are the final inputs for transformer training.

## SUMMARY

- **"conversation"** was selected as the main model input because sentiment is expressed in the dialogue.
- Missing values and duplicate rows were checked to improve data quality.
- Sentiment labels were encoded numerically for classification.
- A stratified validation split was created because the dataset is imbalanced.
- Only light text normalization was applied, since transformers should receive text close to its original form.
- A pretrained DistilBERT tokenizer was used instead of manual vocabulary building.

## Save processed arrays

The final tokenized arrays are saved so the model-training notebook can load them directly without repeating preprocessing.

In [20]:
np.save("X_train_input_ids_customer_only.npy", X_train_input_ids_customer_only)
np.save("X_train_attention_mask_customer_only.npy", X_train_attention_mask_customer_only)
np.save("X_val_input_ids_customer_only.npy", X_val_input_ids_customer_only)
np.save("X_val_attention_mask_customer_only.npy", X_val_attention_mask_customer_only)
np.save("X_test_input_ids_customer_only.npy", X_test_input_ids_customer_only)
np.save("X_test_attention_mask_customer_only.npy", X_test_attention_mask_customer_only)

np.save("y_train_customer_only.npy", y_train_customer_only)
np.save("y_val_customer_only.npy", y_val_customer_only)
np.save("y_test_customer_only.npy", y_test_customer_only)

print("Processed transformer arrays were saved successfully.")

Processed transformer arrays were saved successfully.
